## 2. 먼저 용어부터 정리

### 2.1 차원 (Dimension)

**정의:** 데이터를 표현하기 위해 필요한 축(특성)의 개수입니다.
텍스트 마이닝에서는 **고유 단어의 개수(어휘 사전의 크기)** 가 곧 차원의 수입니다. 단어가 10만 개면 데이터는 10만 차원의 공간에 존재하게 됩니다.

### 2.2 차원의 저주 (Curse of Dimensionality)

**정의:** 차원(단어 수)이 늘어날수록 데이터 사이의 빈 공간이 기하급수적으로 커져서, 데이터들이 서로 너무 멀리 떨어져 의미 있는 패턴(유사도)을 찾기 어려워지는 현상입니다.
- 예를 들어, 10만 차원의 DTM은 99.9%가 `0`으로 채워지게 되어 머신러닝 모델이 학습할 '신호'보다 '빈 공간(노이즈)'이 압도적으로 많아집니다.

### 2.3 주성분 분석 (PCA, Principal Component Analysis)

**정의:** 데이터의 특성을 가장 잘 설명하는(분산이 가장 큰) 새로운 축을 찾아내어 고차원의 데이터를 저차원으로 압축하는 기법입니다. 

### 2.4 잠재 의미 분석 (LSA, Latent Semantic Analysis)

**정의:** 텍스트 데이터를 차원 축소하여, 표면적인 단어 자체(단어의 형태)가 아니라 단어들 이면에 숨어있는 **'잠재적 의미(주제)'** 를 끌어내는 기법입니다. LSA는 수학적으로 절단된 특이값 분해(Truncated SVD)를 사용합니다.

## 3. 차원 축소가 왜 필요한가? (동의어 문제)

단순히 카운트(DTM)나 TF-IDF만 쓰면 모델은 **동의어(Synonym)** 를 전혀 이해하지 못합니다.

**문서 예제:**
- 문서 A: "나는 자동차를 샀다"
- 문서 B: "나는 승용차를 샀다"

어휘 사전: [나는, 자동차를, 승용차를, 샀다]
- 문서 A 벡터: $[1, 1, 0, 1]$
- 문서 B 벡터: $[1, 0, 1, 1]$

**문제점:** '자동차'와 '승용차'는 사실상 같은 의미이지만, DTM에서는 완전히 독립적인 다른 차원(축)으로 취급됩니다. 따라서 이 두 문서의 코사인 유사도를 계산하면 의미가 같은데도 불구하고 유사도가 상당히 낮게 나옵니다. 

차원 축소를 수행하면, '자동차' 차원과 '승용차' 차원이 묶여 **'탈것(Vehicle)'이라는 하나의 잠재적 의미 차원**으로 압축되므로 이 문제가 해결됩니다.

## 4. 직관적인 PCA 예제 (2차원 → 1차원)

텍스트로 넘어가기 전에 직관적인 물리적 예시로 PCA를 이해해봅시다.

우리가 잰 사람들의 데이터가 있습니다.
- 특성 1(X축): **왼쪽 다리 길이**
- 특성 2(Y축): **오른쪽 다리 길이**

데이터 점들을 2차원 그래프에 찍어보면 어떨까요? 왼쪽 다리가 긴 사람은 오른쪽 다리도 길기 때문에, 점들이 우상향하는 얇은 대각선(하나의 선) 모양으로 뭉쳐 있을 것입니다.

**PCA의 작동 방식:**
1. 데이터가 가장 넓게 퍼져있는(분산이 큰) 방향으로 대각선 축(주성분 1)을 새롭게 긋습니다. 이 축의 이름은 **'다리 전체 길이'** 가 됩니다.
2. 기존의 X축(왼쪽 다리)과 Y축(오른쪽 다리)이라는 2차원 데이터를 버리고, 새로 그은 대각선 축(주성분 1)이라는 1차원 데이터로 바꿉니다.

**결과:** 2차원 데이터가 1차원으로 압축되었지만, 정보의 손실은 거의 없습니다! (어차피 양쪽 다리 길이는 비슷하기 때문입니다.) 
이것이 동의어('자동차', '승용차')가 하나의 주성분('탈것')으로 묶이는 원리입니다.

## 5. LSA와 특이값 분해 (SVD)

LSA는 이 개념을 텍스트 DTM 행렬에 적용하는 알고리즘이며, 수학적으로 **특이값 분해(Singular Value Decomposition)**를 사용합니다.

**SVD의 기본 수식:**
어떤 행렬 $A$ (원본 DTM 행렬)는 세 개의 행렬의 곱으로 분해할 수 있습니다.
$A = U \times \Sigma \times V^T$

- $A$: 원본 문서-단어 행렬 (예: 문서 1만 개 $\times$ 단어 5만 개)
- $U$: 문서와 '잠재 의미(주제)' 간의 관계를 나타내는 행렬
- $\Sigma$: 각 잠재 의미(주제)의 중요도를 나타내는 대각 행렬
- $V^T$: 단어와 '잠재 의미(주제)' 간의 관계를 나타내는 행렬

### 5.1 Truncated SVD (절단된 SVD)

원본 SVD를 그대로 곱하면 다시 원래의 비대한 행렬 $A$가 됩니다. 차원을 축소하려면 중요도 행렬인 $\Sigma$에서 **가장 중요도가 큰 (값이 큰) 상위 $k$개의 토픽만 남기고 나머지는 가위로 잘라버립니다 (Truncate).**

$A' = U_k \times \Sigma_k \times V_k^T$

예를 들어 $k=100$으로 설정했다면:
- 원래 5만 개의 단어 축을 가졌던 문서들이 이제 **단 100개의 잠재 의미(Topic) 축**으로 압축됩니다.
- 이렇게 만들어진 행렬 $A'$의 벡터를 사용해 코사인 유사도를 구하면, '자동차'와 '승용차'가 같은 토픽으로 묶여 있기 때문에 문서 A와 문서 B의 유사도가 매우 높게 계산됩니다.